# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [17]:
task_type = "Classification"

why_this_type = """
Last week's frame (Lane 1: Ranking Signal Analysis) asked "which signals are
associated with visibility, clicks, engagement, or movement" -- that's a signal-
analysis question, not yet a task type a model can be trained on. Of the four
concrete options, Classification fits best: the underlying question becomes
"will this page's clicks decline in the next 30-day window or not?" -- a yes/no
question about an observed future outcome, per the framing skill's mapping table
("Will this one decline / recover?" -> Classification -> yes/no label from an
OBSERVED outcome -> ROC-AUC, precision/recall vs base rate).

I'm not picking Clustering because I already have signal evidence (position tier
-> CTR) I want to test predictively, not just group pages by similarity. I'm not
picking Ranking/scoring yet because that's the Lane 2 job (build a queue with
scores) -- this week is about proving the underlying pattern is learnable at all
before I build a queue on top of it.
"""
print(task_type)
print(why_this_type)


Classification

Last week's frame (Lane 1: Ranking Signal Analysis) asked "which signals are
associated with visibility, clicks, engagement, or movement" -- that's a signal-
analysis question, not yet a task type a model can be trained on. Of the four
concrete options, Classification fits best: the underlying question becomes
"will this page's clicks decline in the next 30-day window or not?" -- a yes/no
question about an observed future outcome, per the framing skill's mapping table
("Will this one decline / recover?" -> Classification -> yes/no label from an
OBSERVED outcome -> ROC-AUC, precision/recall vs base rate).

I'm not picking Clustering because I already have signal evidence (position tier
-> CTR) I want to test predictively, not just group pages by similarity. I'm not
picking Ranking/scoring yet because that's the Lane 2 job (build a queue with
scores) -- this week is about proving the underlying pattern is learnable at all
before I build a queue on top of it.



## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [18]:
target_definition = """
Target: declined -- a binary label, 1 if a page's clicks fell from its prior
30-day window to its most recent 30-day window, 0 otherwise.

declined = 1 if clicks_last_30d < clicks_prev_30d else 0

This is an OBSERVED outcome, not a rule someone defined for me: clicks_last_30d
and clicks_prev_30d are both raw measured counts already in the starter CSV, from
two non-overlapping trailing windows. I compute the comparison myself -- I do NOT
use trend_direction or trend_pct, since those come from the pipeline's own
is_declining_label definition (the leakage trap flagged in the data skill). Using
them would mean my model learns to copy an existing rule, not discover anything.

I also restrict to pages with clicks_prev_30d >= 5. Below that, "clicks fell from
2 to 1" is mostly noise, not a real decline -- the threshold keeps the label
meaningful rather than dominated by near-zero-traffic pages.
"""
print(target_definition)



Target: declined -- a binary label, 1 if a page's clicks fell from its prior
30-day window to its most recent 30-day window, 0 otherwise.

declined = 1 if clicks_last_30d < clicks_prev_30d else 0

This is an OBSERVED outcome, not a rule someone defined for me: clicks_last_30d
and clicks_prev_30d are both raw measured counts already in the starter CSV, from
two non-overlapping trailing windows. I compute the comparison myself -- I do NOT
use trend_direction or trend_pct, since those come from the pipeline's own
is_declining_label definition (the leakage trap flagged in the data skill). Using
them would mean my model learns to copy an existing rule, not discover anything.

I also restrict to pages with clicks_prev_30d >= 5. Below that, "clicks fell from
2 to 1" is mostly noise, not a real decline -- the threshold keeps the label
meaningful rather than dominated by near-zero-traffic pages.



## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [19]:
metric_choice = """
Metric: ROC-AUC as the headline number, with recall on the declined==1 class
reported alongside it -- both checked against the base rate, per the framing
skill's classification row (ROC-AUC, precision/recall vs base rate).

Why: this is a triage tool, not a guarantee -- a strategist would rather review a
few pages that turn out fine than miss a real decline, so recall on the positive
class matters as much as overall accuracy. ROC-AUC is threshold-independent, so
it tells me if the model ranks decliners above non-decliners at all before I
commit to a specific cutoff. Comparing to the base rate matters because if 63% of
eligible pages already declined (see below), a model has to beat "always guess
decline" to be worth anything.
"""
print(metric_choice)



Metric: ROC-AUC as the headline number, with recall on the declined==1 class
reported alongside it -- both checked against the base rate, per the framing
skill's classification row (ROC-AUC, precision/recall vs base rate).

Why: this is a triage tool, not a guarantee -- a strategist would rather review a
few pages that turn out fine than miss a real decline, so recall on the positive
class matters as much as overall accuracy. ROC-AUC is threshold-independent, so
it tells me if the model ranks decliners above non-decliners at all before I
commit to a specific cutoff. Comparing to the base rate matters because if 63% of
eligible pages already declined (see below), a model has to beat "always guess
decline" to be worth anything.



## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

import os
print(os.getcwd())
print(os.path.exists("../../data/raw/content_refresh_anonymized.csv"))

In [20]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Unit of analysis: one row = one content item (page), evaluated over its own
# trailing prev-30d vs last-30d window pair. Restrict to pages with enough prior
# volume for the label to mean something.
eligible = df[df["clicks_prev_30d"] >= 5].copy()
eligible["declined"] = (eligible["clicks_last_30d"] < eligible["clicks_prev_30d"]).astype(int)

print(f"Eligible rows (clicks_prev_30d >= 5): {len(eligible):,} "
      f"({len(eligible)/len(df):.1%} of the full 30,000-row starter set)")
print(f"Base rate of declined==1: {eligible['declined'].mean():.1%}")

# Safe, non-leaking feature columns for this unit of analysis (excludes
# trend_direction/trend_pct, product scores, and raw ids)
feature_cols = [
    "content_id", "position_tier", "engagement_rate", "scroll_rate",
    "word_count", "content_type", "freshness_tier", "days_since_last_update",
    "clicks_prev_30d", "impressions_prev_30d", "declined",
]
preview = eligible[feature_cols].head(10)
print(preview.to_string())
preview


FileNotFoundError: [Errno 2] No such file or directory: '../../data/raw/content_refresh_anonymized.csv'

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [ ]:
why_ml = """
A single if-statement rule -- e.g. "flag pages where engagement_rate < X" --
can't capture what last week's EDA already showed: position tier alone swings
mean CTR by ~18x (2.76% at top_3 vs 0.15% at deep), and missingness itself tracks
content_type. Decline risk is plausibly driven by several signals moving together
-- position, freshness, word count, content type, prior volume -- not any single
column crossing one threshold. A page can look fine on any one signal and still
be declining because of how several combine (e.g. aging content that also sits at
a weak position tier). That's the "many signals, tangled" case the framing skill
calls out as where ML earns its place over a hand-written rule -- a simple
classifier (like a decision tree or logistic regression) can weigh and combine
those signals in a way one if-statement can't, and its feature importances then
tell me WHICH signals actually mattered, closing the loop back to Lane 1's
original signal-analysis question.
"""
print(why_ml)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.